# Anforderungen installieren

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Setup & Imports

In [4]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime, date, timedelta
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from io import StringIO
import time 

# Verzeichnisse
DATA_DIR = "./data"
REPORT_DIR = "./reports"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

today = datetime.now().strftime("%Y-%m-%d")
CSV_PATH = os.path.join(DATA_DIR, f"egid_buildings_{today}.csv")
REPORT_PATH = os.path.join(REPORT_DIR, f"updatereport_{today}.txt")


# Datenquelle & Download

In [5]:
# === Setup ===
datum = date.today().isoformat()
os.makedirs("data/tmp_chunks", exist_ok=True)
final_csv_path = f"data/all_buildings_before_1990_{datum}.csv"

base_url = "https://data.egid.ch/current.csv"
batch_size = 1000

# Kleine Ersatzfunktion für sleep ohne time-Modul
def wait_seconds(seconds):
    end_time = datetime.now() + timedelta(seconds=seconds)
    while datetime.now() < end_time:
        pass  # einfache Warte-Schleife (busy wait)

# Session mit automatischem Retry
session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[500, 502, 503, 504],
)
session.mount("https://", HTTPAdapter(max_retries=retries))

# === Abfrage ohne ORDER BY (wird am Ende lokal sortiert)
query = "SELECT * FROM building WHERE GBAUJ < 1990 OR GBAUJ = ''"
last_egid = 0
chunk_counter = 0
total_rows = 0

print("⬇️ Lade Gebäudedaten (<1990 oder ohne Baujahr) aus data.egid.ch ...\n")

while True:
    sql = f"{query} AND EGID > {last_egid} ORDER BY EGID ASC LIMIT {batch_size}"
    url = f"{base_url}?format=csv&sql={sql}"

    try:
        response = session.get(url, timeout=120)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Fehler bei EGID > {last_egid}: {e}")
        print("⏳ Warte 15 Sekunden und versuche erneut ...")
        wait_seconds(15)
        continue

    chunk = pd.read_csv(StringIO(response.text), sep=",", on_bad_lines="skip")

    if chunk.empty:
        print("\n✅ Keine weiteren Zeilen – Download abgeschlossen.")
        break

    count = len(chunk)
    last_egid = chunk["EGID"].max()
    total_rows += count
    chunk_counter += 1

    # Temporär speichern
    chunk_path = f"data/tmp_chunks/chunk_{chunk_counter:05d}.csv"
    chunk.to_csv(chunk_path, index=False)
    print(f"  → Chunk {chunk_counter:>3}: {count:,} Zeilen (bis EGID={last_egid})")

    # Kurze Pause, um Server zu schonen
    wait_seconds(0.5)

print(f"\n📊 Gesamt: {total_rows:,} Zeilen heruntergeladen. Kombiniere CSVs ...")

# === Alle Chunks zusammenführen ===
all_chunks = []
for f in sorted(os.listdir("data/tmp_chunks")):
    part = pd.read_csv(os.path.join("data/tmp_chunks", f))
    all_chunks.append(part)
df = pd.concat(all_chunks, ignore_index=True)

# Lokal sortieren (wenn nötig)
df = df.sort_values("EGID")

df.to_csv(final_csv_path, index=False)
print(f"✅ Finale Datei gespeichert: {final_csv_path}")
print(f"📦 Enthält {len(df):,} Datensätze")

#


⬇️ Lade Gebäudedaten (<1990 oder ohne Baujahr) aus data.egid.ch ...

  → Chunk   1: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   2: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   3: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   4: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   5: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   6: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   7: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   8: 1,000 Zeilen (bis EGID=1000916)
  → Chunk   9: 1,000 Zeilen (bis EGID=1000916)


KeyboardInterrupt: 

# Updatereport erzeugen

In [ ]:
cur.execute("""
SELECT egid, gemeinde, kanton
FROM allegebaeude
WHERE geaendert_von = 'user';
""")

user_locked = cur.fetchall()

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(f"Update-Report für {today}\n")
    f.write("=" * 60 + "\n\n")
    f.write("Gebäude, die wegen Useränderungen NICHT überschrieben wurden:\n\n")
    if user_locked:
        for row in user_locked:
            f.write(f"EGID {row[0]} – {row[1]}, {row[2]}\n")
    else:
        f.write("Keine User-geschützten Datensätze gefunden.\n")

print(f"📄 Report erstellt: {REPORT_PATH}")

cur.close()
conn.close()
